# SpaceX Launch Data Analysis — Løsningsforslag

Komplett løsning for Practice Exercise (50 poeng).

## Part 1: Data Acquisition (10 poeng)

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Hent launches
url_launches = 'https://api.spacexdata.com/v4/launches'
url_rockets = 'https://api.spacexdata.com/v4/rockets'

resp_launches = requests.get(url_launches)
resp_rockets = requests.get(url_rockets)

print(f'Launches status: {resp_launches.status_code}')
print(f'Rockets status:  {resp_rockets.status_code}')

launches_json = resp_launches.json()
rockets_json = resp_rockets.json()

print(f'Antall launches: {len(launches_json)}')
print(f'Antall raketter: {len(rockets_json)}')

## Part 2: DataFrame Creation & Cleaning (10 poeng)

### df_launches

In [ ]:
# Ekstraher relevante felter
df_launches = pd.DataFrame([{
    'launch_id': l['id'],
    'mission_name': l['name'],
    'date_utc': l['date_utc'],
    'rocket_id': l['rocket'],
    'success': l['success'],
    'upcoming': l['upcoming'],
    'failures': l['failures']
} for l in launches_json])

# Filtrer bort upcoming
df_launches = df_launches[df_launches['upcoming'] == False].copy()

# Konverter dato og ekstraher år
df_launches['date_utc'] = pd.to_datetime(df_launches['date_utc'])
df_launches['launch_year'] = df_launches['date_utc'].dt.year

print(f'Shape: {df_launches.shape}')
display(df_launches.head())

### df_rockets

In [ ]:
df_rockets = pd.DataFrame([{
    'rocket_id': r['id'],
    'rocket_name': r['name'],
    'cost_per_launch': r['cost_per_launch'],
    'expected_success_rate_pct': r['success_rate_pct']
} for r in rockets_json])

display(df_rockets)

## Part 3: Data Transformation & Merging (10 poeng)

In [ ]:
# Merge på rocket_id
df_merged = df_launches.merge(df_rockets, on='rocket_id', how='left')

# Ekstraher failure_reason
df_merged['failure_reason'] = df_merged['failures'].apply(
    lambda x: x[0]['reason'] if isinstance(x, list) and len(x) > 0 else 'None'
)

# Dropp overflødige kolonner
df_merged.drop(columns=['upcoming', 'failures', 'rocket_id'], inplace=True)

print(f'Shape: {df_merged.shape}')
display(df_merged.head())

## Part 4: Insights from the Data (20 poeng)

### 4.1 Launches per år

In [ ]:
launches_per_year = df_merged.groupby('launch_year').size()
print(launches_per_year)
print(f'\nÅret med flest launches: {launches_per_year.idxmax()} ({launches_per_year.max()} launches)')

### 4.2 Faktisk success rate per rakett

In [ ]:
success_rate = df_merged.groupby('rocket_name')['success'].apply(
    lambda x: x.sum() / len(x) * 100
).round(2)
print(success_rate)

### 4.3 Total kostnad for feilede launches

In [ ]:
failed = df_merged[df_merged['success'] == False]
total_failed_cost = failed['cost_per_launch'].sum()
print(f'Total kostnad for feilede launches: ${total_failed_cost:,.0f}')

### 4.4 Vanligste failure reason

In [ ]:
failed_reasons = df_merged[df_merged['failure_reason'] != 'None']['failure_reason']
print(failed_reasons.value_counts())

### 4.5 Heldigste måned

In [ ]:
df_merged['launch_month'] = df_merged['date_utc'].dt.month
monthly_success = df_merged.groupby('launch_month')['success'].mean() * 100
print(monthly_success.round(2))
print(f'\nHeldigste måned: {monthly_success.idxmax()} ({monthly_success.max():.1f}%)')

### 4.6 Tid på døgnet med flest launches

In [ ]:
df_merged['launch_hour'] = df_merged['date_utc'].dt.hour

def categorize_time(hour):
    if 0 <= hour <= 5:
        return 'Night'
    elif 6 <= hour <= 11:
        return 'Morning'
    elif 12 <= hour <= 17:
        return 'Afternoon'
    else:
        return 'Evening'

df_merged['time_of_day'] = df_merged['launch_hour'].apply(categorize_time)
print(df_merged['time_of_day'].value_counts())

### 4.7 Lengste streak med suksessfulle launches

In [ ]:
df_sorted = df_merged.sort_values('date_utc').reset_index(drop=True)

# Lag grupper der success endrer seg
df_sorted['streak_group'] = (df_sorted['success'] != df_sorted['success'].shift()).cumsum()

# Finn lengste streak med success == True
streaks = df_sorted[df_sorted['success'] == True].groupby('streak_group').size()
longest = streaks.max()
print(f'Lengste streak med suksessfulle launches: {longest}')

### 4.8 Total pengebruk per år

In [ ]:
cost_per_year = df_merged.groupby('launch_year')['cost_per_launch'].sum()
print(cost_per_year.apply(lambda x: f'${x:,.0f}'))

### 4.9 Stacked bar chart: Success vs Failed per rakett

In [ ]:
pivot = df_merged.groupby(['rocket_name', 'success']).size().unstack(fill_value=0)
pivot.columns = ['Failed', 'Successful']

pivot.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], figsize=(10, 6))
plt.title('Successful vs Failed Launches per Rocket')
plt.xlabel('Rocket Name')
plt.ylabel('Number of Launches')
plt.legend(title='Outcome')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.10 Kumulativ launch-graf over tid

In [ ]:
df_sorted = df_merged.sort_values('date_utc')
df_sorted['cumulative_launches'] = range(1, len(df_sorted) + 1)

plt.figure(figsize=(12, 6))
plt.plot(df_sorted['date_utc'], df_sorted['cumulative_launches'], linewidth=2)
plt.title('Cumulative SpaceX Launches Over Time')
plt.xlabel('Date')
plt.ylabel('Total Launches')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()